# Notebook 2 — Scattering + SVM y PCA generativo sobre MNIST

Monta el pipeline de clasificación y lo contrasta con la Tabla 4 de Bruna y
Mallat. Se evalúan cuatro combinaciones en tres tamaños de entrenamiento:

| Descriptor | Clasificadores |
|---|---|
| píxeles crudos (784 dim) | SVM RBF |
| scattering orden 1 (400 dim) | SVM RBF, PCA afín |
| scattering orden 2 (3472 dim) | SVM RBF, PCA afín |

**Normalizaciones.** El paper usa una distinta por clasificador, y no es
cosmético: la energía se reparte muy desigualmente entre caminos, así que sin
ecualizar, los de orden 1 aplastan a los de orden 2. La SVM lleva los
coeficientes a $[-1,1]$; el PCA usa la ecuación (19), dividiendo cada camino
$p$ por $\max_i \|S[p]X_i\|$ sobre las señales de entrenamiento.

**Selección de hiperparámetros.** Siempre por validación cruzada *dentro* de
las $n$ muestras de entrenamiento. El test no interviene en ninguna decisión.

In [ ]:
import sys

sys.path.insert(0, "..")

import numpy as np
import pandas as pd

from src.data import load_mnist, pad_to_square, stratified_subsample
from src.evaluation import evaluate_affine_pca, evaluate_svm, save_results
from src.features import PathNormalizer, cached_scattering, flatten, select_orders
from src.plotting import plot_replication_comparison, save_figure, use_paper_style
from src.repro import environment_report, set_seed
from src.scattering import build_scattering, verify_paths

J, L, SIZE, SEED = 3, 8, 32, 0
TRAIN_SIZES = [300, 1000, 5000]

set_seed(SEED)
use_paper_style()
environment_report().to_dict()

In [ ]:
x_train, y_train, x_test, y_test = load_mnist()
train_padded = pad_to_square(x_train, SIZE)
test_padded = pad_to_square(x_test, SIZE)

scattering = build_scattering(J=J, L=L, shape=(SIZE, SIZE), max_order=2)
paths = verify_paths(scattering)

features_train = cached_scattering(train_padded, scattering, "mnist_train", J=J, L=L, size=SIZE, order=2)
features_test = cached_scattering(test_padded, scattering, "mnist_test", J=J, L=L, size=SIZE, order=2)

pixels_train = x_train.reshape(len(x_train), -1)
pixels_test = x_test.reshape(len(x_test), -1)

print(f"scattering train: {features_train.shape}")
print(f"dimensión del descriptor por orden: "
      f"1 -> {select_orders(features_train[:1], paths, 1).size}, "
      f"2 -> {select_orders(features_train[:1], paths, 2).size}")

## Barrido de experimentos

Una sola semilla en este notebook: aquí se valida que el pipeline reproduce al
paper. Las barras de error con varias semillas son cosa del Notebook 4, donde
la variabilidad sí es el objeto de estudio.

In [ ]:
results = []

for n in TRAIN_SIZES:
    subset = stratified_subsample(y_train, n, seed=SEED)
    labels = y_train[subset]

    results.append(
        evaluate_svm(pixels_train[subset], labels, pixels_test, y_test, "pixels", SEED)
    )

    for order in (1, 2):
        train_sub = select_orders(features_train[subset], paths, order)
        test_sub = select_orders(features_test, paths, order)

        # Ecuación (19) para el PCA; escalado a [-1, 1] (dentro del pipeline) para la SVM.
        normalizer = PathNormalizer().fit(train_sub)
        pca_train = flatten(normalizer.transform(train_sub))
        pca_test = flatten(normalizer.transform(test_sub))

        results.append(
            evaluate_affine_pca(pca_train, labels, pca_test, y_test, f"scat{order}", SEED)
        )
        results.append(
            evaluate_svm(flatten(train_sub), labels, flatten(test_sub), y_test, f"scat{order}", SEED)
        )

    print(f"n={n} completado")

save_results(results, "02_replica_mnist")

table = pd.DataFrame(
    [
        {
            "n": r.train_size,
            "descriptor": r.descriptor,
            "modelo": r.model,
            "error %": round(100 * r.error_rate, 2),
            "dim": r.n_features,
            "segundos": round(r.fit_seconds, 1),
        }
        for r in results
    ]
)
table

## Contraste con la Tabla 4 del paper

In [ ]:
reference = pd.read_csv("../results/reference/bruna_mallat_2013_table4.csv", comment="#")

SERIES = {
    "pixels_svm": ("pixels", "svm", "pixels_svm"),
    "scat1_pca": ("scat1", "affine_pca", "scat1_pca"),
    "scat2_pca": ("scat2", "affine_pca", "scat2_pca"),
    "scat2_svm": ("scat2", "svm", "scat2_svm"),
}

ours = {
    key: {
        r.train_size: 100 * r.error_rate
        for r in results
        if r.descriptor == descriptor and r.model == model
    }
    for key, (descriptor, model, _) in SERIES.items()
}

paper = {
    key: dict(zip(reference["train_size"], reference[column]))
    for key, (_, _, column) in SERIES.items()
}

comparison = pd.DataFrame(
    [
        {
            "n": n,
            "serie": key,
            "nuestro %": round(ours[key][n], 2),
            "paper %": paper[key][n],
            "diferencia": round(ours[key][n] - paper[key][n], 2),
        }
        for key in SERIES
        for n in sorted(ours[key])
    ]
)
comparison

In [ ]:
fig = plot_replication_comparison(ours, paper)
save_figure(fig, "fig06_replica_tabla4")

## Lectura

**La réplica funciona.** Las diferencias con el paper son de décimas, pese a que
Kymatio no implementa la *reduced cosine scattering representation* de la
sección 3.3, que conserva solo la mitad de baja frecuencia de los coeficientes.
No cabía esperar coincidencia exacta y no la hay; sí coinciden las magnitudes y,
sobre todo, el comportamiento cualitativo.

**El PCA gana a la SVM con pocos datos.** En los tres tamaños medidos, el
clasificador generativo de espacios afines queda por debajo en error que la SVM
sobre el mismo descriptor. El paper explica por qué y su argumento es de
compromiso sesgo-varianza: con pocas muestras por clase, los términos de
varianza al estimar covarianzas cruzadas entre clases dominan al sesgo del
modelo rígido. Al crecer $n$ el compromiso se invierte, y en la Tabla 4 la SVM
acaba adelantando entre $n=10^4$ y $n=2\cdot10^4$.

Conviene notar que ese argumento es **el mismo** que sostiene la hipótesis
central del trabajo, aplicado un nivel más arriba: scattering es a CNN lo que
PCA generativo es a SVM. Menos parámetros que estimar, ventaja cuando los datos
escasean, desventaja cuando sobran.

**El orden 2 aporta.** El error baja de forma consistente al pasar de orden 1 a
orden 2, lo que confirma con nuestros propios números el argumento de energía
del Notebook 1 y descarta la lectura de que los coeficientes de segundo orden
sean redundantes.

**Diagnóstico del modelo afín.** Los valores de $\sigma_d^2$ y $\lambda_d$
guardados en `results/02_replica_mnist.json` reproducen la tendencia de la Tabla
5: al crecer $n$, la dimensión $d$ elegida por validación cruzada sube, el error
de aproximación intra-clase baja y la separación entre clases mejora. Esa es la
explicación mecánica de por qué el clasificador mejora con más datos, y no solo
la constatación de que mejora.